In [ ]:
import os
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import bar_chart_race as bcr
from collections import defaultdict
from datetime import datetime
from pathlib import Path

In [ ]:
def patch_pandas_fillna():
    # Pobieramy referencje do ORYGINALNYCH metod, zanim je nadpiszemy
    # To kluczowe, aby uniknąć pętli (RecursionError)
    original_df_fillna = pd.DataFrame.fillna
    original_series_fillna = pd.Series.fillna

    def universal_fillna(self, value=None, method=None, **kwargs):
        # 1. Jeśli biblioteka prosi o ffill/bfill, używamy nowych metod Pandasa
        if method == 'ffill':
            return self.ffill()
        if method == 'bfill':
            return self.bfill()
        
        # 2. Jeśli to standardowe wywołanie (bez method), używamy oryginału.
        # Sprawdzamy, czy 'self' to DataFrame czy Series, i wywołujemy właściwy oryginał.
        if isinstance(self, pd.DataFrame):
            return original_df_fillna(self, value=value, **kwargs)
        return original_series_fillna(self, value=value, **kwargs)

    # Nadpisujemy metody w klasach Pandasa
    pd.DataFrame.fillna = universal_fillna
    pd.Series.fillna = universal_fillna

# Wywołujemy patch
patch_pandas_fillna()

In [ ]:
# --- 0. PATCHING ---

# problem with bar_chart_race library 
def fix_fillna(cls):
    orig_func = cls.fillna
    def new_fillna(self, value=None, method=None, **kwargs):
        # 1. Handle explicit 'ffill' or 'bfill' calls
        if method == 'ffill':
            return self.ffill()
        if method == 'bfill':
            return self.bfill()
        
        # 2. If method is None or something else, call orig_func WITHOUT method argument
        # We must pop it from kwargs if it exists or just call without it
        # because Pandas 2.1+ crashes if 'method' is present at all.
        return orig_func(self, value=value, **kwargs)
        
    cls.fillna = new_fillna

# Apply to both classes to cover all library edge cases
fix_fillna(pd.DataFrame)
fix_fillna(pd.Series)

In [ ]:
# --- 1. DATA LOADING AND PARSING ---

def load_sequence_lengths(file_path):
    """
    Parses pdb_seqres.txt to extract the length of protein chains.
    Only entries with 'mol:protein' are considered.
    """
    protein_lengths = {}
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith(">") and "mol:protein" in line:
                # Format: >PDBID_CHAIN mol:protein length:N ...
                parts = line.split()
                # Extract 4-character PDB ID
                pdb_id = parts[0][1:5].upper()
                
                # Extract length from 'length:N'
                length_part = [p for p in parts if p.startswith("length:")][0]
                length = int(length_part.split(":")[1])
                
                # Sum lengths if PDB ID has multiple protein chains
                protein_lengths[pdb_id] = protein_lengths.get(pdb_id, 0) + length
    return protein_lengths

def load_experimental_metadata(file_path, protein_ids):
    """
    Parses entries.idx for dates, resolution, and experimental methods.
    Filters data based on the provided list of protein-only PDB IDs.
    """
    metadata = {}
    with open(file_path, 'r') as f:
        # Skip header lines
        for _ in range(2): next(f)
        
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 8:
                continue
                
            pdb_id = parts[0].upper()
            
            # Only process if it's a protein entry found in seqres
            if pdb_id not in protein_ids:
                continue
                
            header_name = parts[1]
            date_str = parts[2]
            resolution_str = parts[6]
            method = parts[7].upper()
            
            # Handle Date (MM/DD/YY) and extract Year
            try:
                dt = datetime.strptime(date_str, "%m/%d/%y")
                year = dt.year if dt.year <= 2026 else dt.year - 100
            except ValueError:
                continue

            # Assign resolution 'r' based on method
            # Per instructions: NMR is set to 2.5 [cite: 14]
            if "NMR" in method:
                r = 2.5
            else:
                try:
                    r = float(resolution_str)
                except ValueError:
                    r = 0.0 # Placeholder for invalid strings
            
            # IMPORTANT : Skip if r is 0 to avoid ZeroDivisionError - those can arrise if we obtain empty row 
            if r <= 0:
                continue
                
            metadata[pdb_id] = {
                'year': year,
                'method': method,
                'r': r,
                'header': header_name
            }
    return metadata

def load_entry_types(file_path):
    """
    Parses pdb_entry_type.txt to identify the molecule type (prot, nuc, etc.).
    Useful for secondary verification.
    """
    entry_types = {}
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 3:
                pdb_id = parts[0].upper()
                mol_type = parts[1] # e.g., 'prot', 'nuc', 'prot-nuc'
                method_type = parts[2] # e.g., 'diffraction', 'nmr'
                entry_types[pdb_id] = {'mol_type': mol_type, 'method_type': method_type}
    return entry_types

# --- Example of how to combine them ---
def process_pdb_data(data_dir):
    # 1. Load sequence lengths first
    seq_path = os.path.join(data_dir, "pdb_seqres.txt")
    protein_lengths = load_sequence_lengths(seq_path)
    
    # 2. Load metadata using IDs from sequences to filter
    idx_path = os.path.join(data_dir, "entries.idx")
    metadata = load_experimental_metadata(idx_path, protein_lengths.keys())
    
    # 3. Combine and calculate the formula sum(n * 1/r)
    combined_results = []
    for pdb_id, info in metadata.items():
        n = protein_lengths[pdb_id]
        r = info['r']
        value = n * (1.0 / r) # No longer throws ZeroDivisionError
        
        combined_results.append({
            'pdb_id': pdb_id,
            'year': info['year'],
            'method': info['method'],
            'value': value,
            'header': info['header']
        })
        
    return pd.DataFrame(combined_results)

In [ ]:
# --- 2. STATIC PIE CHART AND CLASSIFICATION ---

def create_pie_chart(df, class_mapping, save: Path | None = None, **kwargs):
    """
    Creates a pie chart. 
    class_mapping: dictionary { 'Category Name': ['Method1', 'Method2'] }
    """
    mapped_data = defaultdict(float)
    skipped_elements = []
    
    # Iterate through all records
    for pdb_id, row in df.iterrows():
        exp = row['method']
        matched = False
        
        # Look for a match in defined classes
        for category, methods in class_mapping.items():
            if any(method in exp for method in methods):
                mapped_data[category] += row['value']
                matched = True
                break
                
        # If no match found, add to skipped elements
        if not matched:
            skipped_elements.append(f"{exp} ({row['header']})")

    # explode graph version 
    # # legend preparation
    # labels = list(mapped_data.keys())
    # values = list(mapped_data.values())
            
    # # Drawing the pie chart
    # fig, ax = plt.subplots(figsize=(12, 8)) # Większy obszar
    # ## plot without labels
    # wedges, texts, autotexts = ax.pie(
    #     values, 
    #     autopct=lambda p: f'{p:.1f}%' if p > 2 else '', 
    #     startangle=140,
    #     pctdistance=0.85, # Procenty nieco dalej od środka
    #     explode=[0.05] * len(labels) # Delikatne rozsuniecie kawałków poprawia czytelność
    # )
    # ## side labels
    # ax.legend(
    #     wedges, 
    #     labels,
    #     title="Metody eksperymentalne",
    #     loc="center left",
    #     bbox_to_anchor=(1, 0, 0.5, 1) # Przesunięcie legendy poza wykres
    # )
    # plt.title('Share of experimental methods in protein structures in PDB $\sum(n \cdot 1/r)$', pad=20)
    # plt.axis('equal')

    # OLD 
    plt.pie(mapped_data.values(), labels=mapped_data.keys(), autopct='%1.1f%%', startangle=140)
    plt.title('Share of experimental methods in protein structures in PDB $\sum(n \cdot 1/r)$')
    plt.axis('equal')
    if save:
        save = Path(save).with_suffix('.png')
        plt.savefig(save)
    
    # Printing skipped elements (unique entries to avoid flooding the console)
    print("\n--- SKIPPED CATEGORIES ---")
    unique_skipped = list(set(skipped_elements))
    for skipped in unique_skipped[:20]: # Show first 20 unique for clarity
        print(f"- {skipped}")
    if len(unique_skipped) > 20:
        print(f"... and {len(unique_skipped) - 20} others.")
        
    plt.show()
    return mapped_data

# --- 3. ANIMATED BARPLOTS (2005 - 2025) ---

def create_animated_barplots(df, class_mapping):
    # Mapping to main categories for the entire DataFrame
    def map_exp(exp):
        for category, methods in class_mapping.items():
            if any(method in exp for method in methods):
                return category
        return "OTHER"
        
    df['category'] = df['method'].apply(map_exp)
    
    # Filter years 2005-2025 and remove "OTHER"
    df_filtered = df[(df['year'] >= 2005) & (df['year'] <= 2025) & (df['category'] != "OTHER")]
    
    # Group by year and category, then calculate cumulative sum
    # PDB data volume grows over time, so we use accumulation
    yearly_data = df_filtered.groupby(['year', 'category'])['value'].sum().unstack(fill_value=0)
    yearly_data_cumsum = yearly_data.cumsum()
    
    # Version 1: Absolute numbers
    bcr.bar_chart_race(
        df=yearly_data_cumsum,
        filename='absolute_growth_2005_2025.gif',
        orientation='h',
        sort='desc',
        n_bars=3,
        title='Data growth in PDB (absolute values $\sum(n \cdot 1/r)$)',
        steps_per_period=10,
        period_length=1000,
        writer='imagemagick'
    )
    
    # Version 2: Percentages (calculating percentage share for each year)
    yearly_data_pct = yearly_data_cumsum.div(yearly_data_cumsum.sum(axis=1), axis=0) * 100
    
    bcr.bar_chart_race(
        df=yearly_data_pct,
        filename='percentage_growth_2005_2025.gif',
        orientation='h',
        sort='desc',
        n_bars=3,
        title='Percentage share of methods in PDB (2005-2025)',
        steps_per_period=10,
        period_length=1000,
        writer='imagemagick'
    )

In [ ]:
# load all data
df = process_pdb_data(data_dir="data/pdb")

classes = {
        'X-RAY DIFFRACTION': ['X-RAY'],
        'ELECTRON MICROSCOPY': ['ELECTRON MICROSCOPY', 'ELECTRON CRYSTALLOGRAPHY', 'CRYO-ELECTRON MICROSCOPY'],
        'NMR': ['NMR'],
        'NEUTRON DIFFRACTION': ['NEUTRON DIFFRACTION'],   # small class (1)
        # 'FIBER DIFFRACTION': ['FIBER DIFFRACTION'],       # small class (1)
        # 'POWDER DIFFRACTION': ['POWDER DIFFRACTION'],     # small class (7)
        # 'SOLUTION SCATTERING': ['SOLUTION SCATTERING ']   # small class (1)
    }


In [ ]:
df['method'].value_counts()

In [ ]:
create_pie_chart(df, class_mapping=classes)

In [ ]:
create_animated_barplots(df, class_mapping=classes)